In [5]:
import numpy as np

In [ ]:
datos_experimentales = np.load('conversion_datos_Exp.npz')

alpha = datos_experimentales['alpha']
delta = datos_experimentales['delta']
tiempos_dias_jd = datos_experimentales['tiempos_dias_jd']
tiempo_central = datos_experimentales['tiempo_Central']


In [12]:
#Contrsucción del vector rho

rho = np.zeros((4, 3))

for i in range(4):
    rho[i, 0] = np.cos(delta[i]) * np.cos(alpha[i])  # x
    rho[i, 1] = np.cos(delta[i]) * np.sin(alpha[i])  # y
    rho[i, 2] = np.sin(delta[i])                     # z

In [ ]:
#Interpolación de los componentes de rho (lambda, mu y nu)

#Como los dias julianos son muy grandes vamos a reescalar para ver si funciona bien c: Revisar

tiempo_relativo = tiempos_dias_jd - tiempos_dias_jd[0]

poly_lambda = np.polyfit(tiempo_relativo, rho[:, 0], 3)
poly_mu = np.polyfit(tiempo_relativo, rho[:, 1], 3)
poly_nu = np.polyfit(tiempo_relativo, rho[:, 2], 3)

tiempo_central_reescalado = tiempo_central - tiempos_dias_jd[0] 

In [18]:
#Para lambda

l_val = np.polyval(poly_lambda, tiempo_central_reescalado)

derivada_1_l = np.polyder(poly_lambda,1)
l_dval = np.polyval(derivada_1_l, tiempo_central_reescalado)

derivada_2_l = np.polyder(poly_lambda,2)
l_ddval = np.polyval(derivada_2_l, tiempo_central_reescalado)

#Para mu

mu_val = np.polyval(poly_mu, tiempo_central_reescalado)

derivada_1_mu = np.polyder(poly_mu,1)
mu_dval = np.polyval(derivada_1_mu, tiempo_central_reescalado)

derivada_2_mu = np.polyder(poly_mu,2)
mu_ddval = np.polyval(derivada_2_mu, tiempo_central_reescalado)

#Para nu

nu_val = np.polyval(poly_nu, tiempo_central_reescalado)

derivada_1_nu = np.polyder(poly_nu,1)
nu_dval = np.polyval(derivada_1_nu, tiempo_central_reescalado)

derivada_2_nu = np.polyder(poly_nu,2)
nu_ddval = np.polyval(derivada_2_nu, tiempo_central_reescalado)

In [ ]:
#l_val, mu_val y nu_val están dentro del rango de -1 a 1.
#l_dval está en el rango de 10e-3, y mu_dval y nu_dval están en el rango de 10e-4.
#l_ddval, mu_ddval y nu_ddval están en el rango de 10e-5. Esto es consistente con lo que se espera para las posiciones y velocidades de un planeta en el sistema solar.

np.float64(-2.4778081356930103e-05)

In [23]:
#Datos de las efemérides del Sol

X = -0.92166819869956
Y = 0.3624168676580381
Z = -1.877615350367358e-5


#Distancia desde el Sol hasta el planeta Tierra (aproximadamente 1UA)
R = np.sqrt(X**2 + Y**2 + Z**2)

In [ ]:
#Cálculo de las matrices D(rho) y D1(rho y R)

matriz_D = np.array([
    [l_val,  l_dval,  l_ddval],
    [mu_val, mu_dval, mu_ddval],
    [nu_val, nu_dval, nu_ddval]
])

# Calcular el determinante D
D = np.linalg.det(matriz_D)
D

#D está dando 4.5923456027e-08. O los datos están muys eguidos entre sí, o estamos haciendo algo mal. Revisar.

#k es la constante de gravitación universal en unidades de UA^3/dia^{2}

k = 0.01720209895
k2 = k**2

matriz_D1 = np.array([
    [l_val, l_dval, X],
    [mu_val, mu_dval, Y],
    [nu_val, nu_dval, Z]
])

#Calcular el determiante D1
D1 = k2* np.linalg.det(matriz_D1)
D1

np.float64(1.3002171451210513e-07)

In [42]:
#Intento preliminar
intento = D1/D * (1/(0.99)**3 - 1/(5.2)**3)
intento

np.float64(2.8977999445141935)

In [ ]:
cos_phi = (l_val*X + mu_val*Y +nu_val*Z)/(R)

# Bucle para hallar r y rho

r_obj = 5.2  # Distancia promedio Sol Y Júpiter en UA
rho_old = 0
tolerancia = 1e-8
max_iter = 50
i = 0

print(f"{'Iter':<5} | {'rho (UA)':<15} | {'r (UA)':<15}")
print("-" * 40)

# Bucle
while i < max_iter:
    rho_new = (D1 / D) * (1/R**3 - 1/r_obj**3)
    
    # Ley de los Cosenos: Hallamos el nuevo r a partir de rho
    # "El signo es positivo porque estamos asumiendo que R es desde el Sol hasta la Tierra"
    r_obj = np.sqrt(rho_new**2 + R**2 + 2 * rho_new * R * cos_phi)
    
    print(f"{i:<5} | {rho_new:<15.8f} | {r_obj:<15.8f}")
    
    # Para cuando rho ya no cambia
    if abs(rho_new - rho_old) < tolerancia:
        print("-" * 40)
        print(f"¡Convergencia lograda en {i} iteraciones!")
        break
        
    rho_old = rho_new
    i += 1

rho_final = rho_new
r_final = r_obj

print(f"\nDistancia Tierra-Júpiter (rho): {rho_final:.6f} UA")
print(f"Distancia Sol-Júpiter (r): {r_final:.6f} UA")

Iter  | rho (UA)        | r (UA)         
----------------------------------------
0     | 2.89459532      | 3.55457597     
1     | 2.85169098      | 3.51282214     
2     | 2.84941624      | 3.51060918     
3     | 2.84929265      | 3.51048894     
4     | 2.84928592      | 3.51048240     
5     | 2.84928556      | 3.51048204     
6     | 2.84928554      | 3.51048202     
7     | 2.84928554      | 3.51048202     
----------------------------------------
¡Convergencia lograda en 7 iteraciones!

Distancia Tierra-Júpiter (rho): 2.849286 UA
Distancia Sol-Júpiter (r): 3.510482 UA


In [50]:
#Finalmente, el vectoor de jupiter es

x_jup = X + rho_final * l_val
y_jup = Y + rho_final * mu_val
z_jup = Z + rho_final * nu_val

vector_r_jupiter = np.array([x_jup, y_jup, z_jup])
vector_r_jupiter

array([-1.68348241,  2.87333691,  1.11078184])

In [51]:
np.linalg.norm(vector_r_jupiter)

np.float64(3.5105575514767637)